In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

In [ ]:
TARGET = "nama_target"
ID_COL = "id"       # ubah menjadi None jika tidak ada
RANDOM_STATE = 42

In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

print("Train:", train.shape)
print("Test :", test.shape)
print("Sample submission:", sample_submission.shape)

display(train.head())
display(train.info())
display(train.describe(include="all").T)

In [ ]:
print("Missing train:")
display(
    train.isna()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

print("Duplicate train:", train.duplicated().sum())

print("Target distribution:")
display(train[TARGET].value_counts())
display(train[TARGET].value_counts(normalize=True))

In [ ]:
X = train.drop(columns=[TARGET])
y = train[TARGET].copy()
X_test = test.copy()

if ID_COL is not None:
    test_id = X_test[ID_COL].copy()
    X = X.drop(columns=[ID_COL])
    X_test = X_test.drop(columns=[ID_COL])

In [ ]:
numeric_features = X.select_dtypes(
    include="number"
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
transformers = []

if numeric_features:
    transformers.append(
        ("num", numeric_pipeline, numeric_features)
    )

if categorical_features:
    transformers.append(
        ("cat", categorical_pipeline, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers
)

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

In [ ]:
model.fit(X_train, y_train)

valid_prediction = model.predict(X_valid)

print(
    "Accuracy:",
    accuracy_score(y_valid, valid_prediction)
)

print(
    "Macro F1:",
    f1_score(
        y_valid,
        valid_prediction,
        average="macro"
    )
)

print(
    classification_report(
        y_valid,
        valid_prediction,
        zero_division=0
    )
)

ConfusionMatrixDisplay.from_predictions(
    y_valid,
    valid_prediction,
    cmap="Blues"
)
plt.show()

In [ ]:
model.fit(X, y)

In [ ]:
test_prediction = model.predict(X_test)

In [ ]:
submission = sample_submission.copy()
submission[TARGET] = test_prediction

if ID_COL is not None and ID_COL in submission.columns:
    submission[ID_COL] = test[ID_COL].values

In [ ]:
assert submission.shape == sample_submission.shape
assert list(submission.columns) == \
       list(sample_submission.columns)
assert submission.isna().sum().sum() == 0
assert len(submission) == len(test)

submission.to_csv("submission.csv", index=False)

display(submission.head())
print("submission.csv berhasil dibuat.")